# Day 3 — Installing Polars and picking the right build

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** [Day 3 — Installing Polars (and the Choices Nobody Mentions)](https://python.plainenglish.io/day-3-installing-polars-and-the-choices-nobody-mentions-f01644f4b249)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [6]:
import polars as pl

print("polars", pl.__version__)

polars 2.0.0


Polars is one `pip`/`uv` install, but there are a few choices worth knowing about: optional extras, CPU builds, thread settings, and the default engine.

## 1. Project setup with uv (run in a terminal, not here)

```bash
uv init --bare --python 3.12
uv add polars pyarrow pandas duckdb ipykernel
uv run python -m ipykernel install --user --name days-of-polars --display-name "Python (days-of-polars)"
```

Optional extras pull in integrations, for example:

```bash
uv add "polars[pyarrow,pandas]"   # interop extras
uv add "polars[gpu]"              # GPU engine (NVIDIA, Linux/WSL)
uv add "polars[all]"              # everything
```

Separate runtime builds exist for very large tables (64-bit row index) and for older CPUs without modern instruction sets. Check the [installation guide](https://docs.pola.rs/user-guide/installation/) for the current names.

## 2. What did I install?

In [7]:
pl.show_versions()

--------Version info---------
Polars:              2.0.0
Build:               22a147de3d2bb2e44b97338a2510816c7105c9f2
Index type:          UInt32
Platform:            Windows-11-10.0.26200-SP0
Python:              3.13.2 (tags/v3.13.2:4f8bb39, Feb  4 2025, 15:23:48) [MSC v.1942 64 bit (AMD64)]
Runtime:             rt32

----Optional dependencies----
Azure CLI            

2.67.0
adbc_driver_manager  <not installed>
altair               <not installed>
azure.identity       <not installed>
boto3                <not installed>
cloudpickle          <not installed>
connectorx           <not installed>
deltalake            <not installed>
fastexcel            <not installed>
fsspec               <not installed>
gevent               <not installed>
google.auth          <not installed>
great_tables         <not installed>
matplotlib           3.11.2
numpy                2.5.4
openpyxl             <not installed>
pandas               3.0.6
polars_cloud         <not installed>
pyarrow              26.0.0
pydantic             <not installed>
pyiceberg            <not installed>
sqlalchemy           <not installed>
torch                <not installed>
xlsx2csv             <not installed>
xlsxwriter           <not installed>


In [8]:
import os

print("thread pool size:", pl.thread_pool_size())
print("CPU count:       ", os.cpu_count())
print("POLARS_MAX_THREADS env:", os.environ.get("POLARS_MAX_THREADS"))

thread pool size: 12
CPU count:        12
POLARS_MAX_THREADS env: None


Polars sizes its thread pool **once, at import**. To change it, set `POLARS_MAX_THREADS` before `import polars` (or in your shell), then restart the kernel.

## 3. Engines in Polars 2.x

`collect()` hands the optimized plan to an **engine**:

| Engine | How it works | Default for |
|---|---|---|
| `"in-memory"` | whole table in RAM, step by step | eager `DataFrame` methods |
| `"streaming"` | data flows through the plan in small batches | lazy `collect()` and `sink_*` (new in 2.0) |
| `"gpu"` | supported operations on an NVIDIA GPU (`polars[gpu]`) | opt-in only |

`engine="auto"` (the default) is not a fourth engine: it means "Polars chooses". In 2.x that is streaming for lazy queries and in-memory for eager ones. Operations that are not implemented for streaming (yet) "fall back to the in-memory engine" ([streaming docs](https://docs.pola.rs/user-guide/concepts/streaming/#inspecting-a-streaming-query)); the result is the same.

You can pin the engine per query or for the whole session:

In [9]:
lf = pl.scan_csv("data/raw/olist_orders_dataset.csv")

streaming_result = lf.select(pl.len()).collect()  # auto -> streaming
in_memory_result = lf.select(pl.len()).collect(engine="in-memory")
print(streaming_result.item(), in_memory_result.item())

# Session-wide (uncomment to use):
# pl.Config.set_engine_affinity("in-memory")

99441 99441


> **Polars 1.x:** `collect()` used the in-memory engine unless you passed `streaming=True` (later `engine="streaming"`). If you upgrade old pipelines, remember that streaming does not guarantee row order for joins and group-bys — sort explicitly.

## 4. Spilling to disk (out-of-core)

Some operations (a sort, for example) must see all the data before producing output. If that state outgrows RAM, Polars 2.0 writes part of it to temporary files on disk and reads it back later: slower, but the query finishes instead of crashing.

Per the [Polars 2.0 release post](https://pola.rs/posts/release-polars-2/): on by default, starts at ~80% of RAM, 64 GB default disk budget, currently covers sorts, window functions and many expressions (joins and group-bys not yet). The Olist data is far too small to spill.

## 5. Display settings worth setting once

In [10]:
orders = pl.read_csv("data/raw/olist_orders_dataset.csv")

with pl.Config(tbl_rows=5, tbl_cols=-1, fmt_str_lengths=40, tbl_width_chars=200):
    print(orders)

shape: (99_441, 8)
┌──────────────────────────┬──────────────────────────┬──────────────┬──────────────────────────┬─────────────────────┬──────────────────────────┬──────────────────────────┬──────────────────────────┐
│ order_id                 ┆ customer_id              ┆ order_status ┆ order_purchase_timestamp ┆ order_approved_at   ┆ order_delivered_carrier_ ┆ order_delivered_customer ┆ order_estimated_delivery │
│ ---                      ┆ ---                      ┆ ---          ┆ ---                      ┆ ---                 ┆ date                     ┆ _date                    ┆ _date                    │
│ str                      ┆ str                      ┆ str          ┆ str                      ┆ str                 ┆ ---                      ┆ ---                      ┆ ---                      │
│                          ┆                          ┆              ┆                          ┆                     ┆ str                      ┆ str                      ┆ str

`pl.Config` works as a context manager (temporary) or with `pl.Config.set_*` calls (for the session).

## Checklist for the post

- Python: 3.13.2
- Polars: 2.0.0
- Thread pool: 12 threads on 10 cores (12 logical)
- Build: index type UInt32, runtime rt32 (standard build)
- OS: Windows 11

## Try it yourself

1. Run `pl.Config.set_tbl_rows(20)` and print `orders` again. Then reset with `pl.Config.restore_defaults()`.
2. Restart with `POLARS_MAX_THREADS=2` and confirm `pl.thread_pool_size()` changed.

## Bonus: CPUs, cores and threads 101

| Term | What it is | This laptop |
|---|---|---|
| CPU (socket) | the physical chip | 1 |
| Core | an independent worker in the chip | 10 (2 P + 8 E) |
| Logical processor | what the OS schedules onto; a Hyper-Threaded core counts as 2 | 12 |
| Software thread | a stream of work a program creates | Polars pool: 12 |

Polars starts one thread per logical processor, so `pl.thread_pool_size()` matches `os.cpu_count()`. In the cloud, a vCPU is usually one logical processor. Polars is single-node: to go bigger, use a bigger machine (memory first), not more machines.
